# Knee: which models add anything beyond ConvNeXt? (OOF ensemble study, CPU only)
Attach as inputs the notebook outputs that hold the OOF files: baseline training (`oof_all.csv`), `oof_v4mix.csv`, `oof_v6tok.csv`, `oof_v6both.csv`, `oof_v6convnext.csv` (any subset; every file named `oof_*.csv` becomes an arm).
No GPU, no internet needed. Runs in about a minute.

Everything is scored the way the earlier notebooks scored it: AUC on the FIXED old labels (confident only: y >= 0.9 or <= 0.1), each model ranked **within its fold**, ensembles = rank mean (what `infer.py` does).
Decision rule used at the end: a member is worth adding to the reference only if it improves macro by more than about 0.003, the 90% bootstrap CI excludes 0, and it helps in at least 4 of 5 folds.

In [1]:
import os, itertools, json
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata

ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
REF = "convnext"                      # the arm every other arm is compared against
OVERRIDE = {}                         # set if a name is found twice, e.g. {"baseline": "/kaggle/input/.../oof_all.csv"}
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
KEY4 = LAB[:4]
NAMES = {"oof_all.csv": "baseline", "oof_v4mix.csv": "v4mix", "oof_v6tok.csv": "v6tok",
         "oof_v6both.csv": "v6both", "oof_v6convnext.csv": "convnext"}
SKIP = {"cache", "train_series", "test_series", "train", "test", "__pycache__", ".git"}

found = {}
for d, dirs, fs in os.walk(ROOT):
    dirs[:] = [x for x in dirs if x not in SKIP]
    for f in fs:
        if f in NAMES or (f.startswith("oof_") and f.endswith(".csv")):
            name = NAMES.get(f, f[4:-4])
            found.setdefault(name, []).append(Path(d) / f)
for n, ps in found.items(): print(f"{n}: {len(ps)} found", *[f"\n   {p}" for p in ps])
paths = {}
for n, ps in found.items():
    if n in OVERRIDE: paths[n] = Path(OVERRIDE[n]); continue
    assert len(ps) == 1, f"'{n}' found {len(ps)} times: set OVERRIDE['{n}'] to the file you want"
    paths[n] = ps[0]
for n, p in OVERRIDE.items(): paths.setdefault(n, Path(p))
assert REF in paths, f"reference arm '{REF}' not found; found: {sorted(paths)}"
print("\narms:", sorted(paths))

v4mix: 1 found 
   /kaggle/input/notebooks/theokorir/knee-v4-mix-5fold/oof_v4mix.csv
v6tok: 1 found 
   /kaggle/input/notebooks/theokorir/knee-v6-tok/oof_v6tok.csv
convnext: 1 found 
   /kaggle/input/notebooks/theokorir/knee-v6-convnext-5fold/oof_v6convnext.csv
v6convnext2: 1 found 
   /kaggle/input/notebooks/theokorir/convnext-5-fold-retrain-seed43/oof_v6convnext2.csv
baseline: 1 found 
   /kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv

arms: ['baseline', 'convnext', 'v4mix', 'v6convnext2', 'v6tok']


In [2]:
need = [f"p_{l}" for l in LAB] + [f"y_{l}" for l in LAB]
raw = {}
for n, p in paths.items():
    d = pd.read_csv(p); miss = [c for c in need if c not in d.columns]
    assert not miss, f"{n}: {p} lacks columns like {miss[:3]}"
    raw[n] = d
base_name = "baseline" if "baseline" in raw else REF          # fixed old labels come from the baseline file
b0 = raw[base_name]
if all("StudyInstanceUID" in d.columns for d in raw.values()):
    ids = b0.StudyInstanceUID.tolist()
    for n, d in raw.items():
        assert set(d.StudyInstanceUID) == set(ids) and d.StudyInstanceUID.is_unique, f"{n}: study ids differ from {base_name}"
        raw[n] = d.set_index("StudyInstanceUID").loc[ids].reset_index()
else:
    assert len({len(d) for d in raw.values()}) == 1, "no StudyInstanceUID column and different lengths"
    print("WARNING: no StudyInstanceUID column, aligned by row order")
N = len(b0)
Y = np.stack([b0[f"y_{l}"].values for l in LAB], 1).astype(float)
for n, d in raw.items():
    dy = np.nanmax(np.abs(np.stack([d[f"y_{l}"].values for l in LAB], 1) - Y))
    if dy > 1e-6: print(f"WARNING {n}: y_ differs from {base_name} (max diff {dy:.3f}); scoring uses {base_name}'s fixed labels anyway")
fold_src = [n for n, d in raw.items() if "fold" in d.columns]
assert fold_src, "no oof file has a 'fold' column"
fold = raw[fold_src[0]].fold.values.astype(int)
for n in fold_src: assert (raw[n].fold.values.astype(int) == fold).all(), f"{n}: fold assignment differs from {fold_src[0]}"
FOLDS = sorted(np.unique(fold)); print("studies", N, "| folds", FOLDS, "| fold sizes", [int((fold == k).sum()) for k in FOLDS])
gold = (b0.is_gold.values.astype(bool) if "is_gold" in b0.columns else np.zeros(N, bool)); print("gold studies:", int(gold.sum()))

def fold_rank(P):                          # rank within each fold, scaled to (0, 1]
    R = np.zeros_like(P)
    for k in FOLDS:
        m = fold == k
        R[m] = np.apply_along_axis(rankdata, 0, P[m]) / m.sum()
    return R
RK = {n: fold_rank(np.stack([d[f"p_{l}"].values for l in LAB], 1).astype(float)) for n, d in raw.items()}
ARMS = sorted(RK, key=lambda n: (n != REF, n))

def _auc(yb, s):
    n1 = int(yb.sum()); n0 = len(yb) - n1
    if n1 < 5 or n0 < 5: return np.nan
    r = rankdata(s); return (r[yb == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def aucs(R, rows=None):                    # per-label AUC on confident old labels, optional row subset / bootstrap sample
    out = np.full(len(LAB), np.nan)
    for j in range(len(LAB)):
        y = Y[:, j] if rows is None else Y[rows, j]; s = R[:, j] if rows is None else R[rows, j]
        m = (y >= 0.9) | (y <= 0.1)
        if m.sum() > 20: out[j] = _auc((y[m] >= 0.9).astype(int), s[m])
    return out
macro = lambda a: np.nanmean(a); key4 = lambda a: np.nanmean(a[:4])
ens = lambda members, w=None: sum((1 if w is None else w[m]) * RK[m] for m in members) / (len(members) if w is None else sum(w[m] for m in members))
single = {n: aucs(RK[n]) for n in ARMS}
print(pd.DataFrame({n: {"macro": macro(a), "KEY-4": key4(a)} for n, a in single.items()}).T.round(4).to_string())

studies 4407 | folds [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] | fold sizes [882, 882, 881, 881, 881]
gold studies: 58
              macro   KEY-4
convnext     0.8197  0.8292
baseline     0.7657  0.7531
v4mix        0.7619  0.7567
v6convnext2  0.8212  0.8336
v6tok        0.7777  0.7668


In [3]:
# ---- all equal-weight rank-mean subsets ----
subsets = [c for r in range(1, len(ARMS) + 1) for c in itertools.combinations(ARMS, r)] if len(ARMS) <= 8 else \
          [(REF,)] + [(REF, n) for n in ARMS if n != REF] + [tuple(m for m in ARMS if m != n) for n in ARMS] + [tuple(ARMS)]
def _perfold(R, k):
    rows = np.where(fold == k)[0]; return aucs(R, rows)
ref_pf = np.array([np.nanmean(_perfold(RK[REF], k)) for k in FOLDS])
rows = []
for c in subsets:
    R = ens(c); a = aucs(R); pf = np.array([np.nanmean(_perfold(R, k)) for k in FOLDS])
    rows.append(dict(members=" + ".join(c), n=len(c), macro=macro(a), key4=key4(a), d_macro=macro(a) - macro(single[REF]),
                     d_key4=key4(a) - key4(single[REF]), folds_better=f"{int((pf > ref_pf).sum())}/{len(FOLDS)}", worst_fold_delta=(pf - ref_pf).min()))
T = pd.DataFrame(rows).sort_values("macro", ascending=False).reset_index(drop=True)
print(f"reference = {REF} alone: macro {macro(single[REF]):.4f}  KEY-4 {key4(single[REF]):.4f}   (seed noise on one fold ~0.01; pooled noise smaller)\n")
print(T.round(4).to_string(index=False))

reference = convnext alone: macro 0.8197  KEY-4 0.8292   (seed noise on one fold ~0.01; pooled noise smaller)

                                          members  n  macro   key4  d_macro  d_key4 folds_better  worst_fold_delta
                           convnext + v6convnext2  2 0.8240 0.8358   0.0043  0.0066          5/5            0.0024
                                      v6convnext2  1 0.8212 0.8336   0.0015  0.0044          4/5           -0.0007
                                         convnext  1 0.8197 0.8292   0.0000  0.0000          0/5            0.0000
                   convnext + v6convnext2 + v6tok  3 0.8187 0.8255  -0.0010 -0.0037          2/5           -0.0039
                convnext + baseline + v6convnext2  3 0.8175 0.8239  -0.0022 -0.0053          1/5           -0.0047
                   convnext + v4mix + v6convnext2  3 0.8171 0.8243  -0.0026 -0.0049          0/5           -0.0044
        convnext + baseline + v6convnext2 + v6tok  4 0.8123 0.8149  -0.0074 -0.0143 

In [4]:
# ---- does each other arm add to the reference?  (paired bootstrap over studies, 90% CI) ----
rng = np.random.default_rng(0); B = 300
boots = [rng.integers(0, N, N) for _ in range(B)]
def boot_delta(A, Bm):
    dm, dk = [], []
    for b in boots:
        x, y = aucs(A, b), aucs(Bm, b); dm.append(macro(x) - macro(y)); dk.append(key4(x) - key4(y))
    return np.percentile(dm, [5, 95]), np.percentile(dk, [5, 95])
rows = []
for n in ARMS:
    if n == REF: continue
    R = ens((REF, n)); a = aucs(R); (m5, m95), (k5, k95) = boot_delta(R, RK[REF])
    pf = np.array([np.nanmean(_perfold(R, k)) for k in FOLDS])
    rows.append(dict(add=n, d_macro=macro(a) - macro(single[REF]), macro_CI=f"[{m5:+.4f},{m95:+.4f}]", d_key4=key4(a) - key4(single[REF]),
                     key4_CI=f"[{k5:+.4f},{k95:+.4f}]", folds_better=f"{int((pf > ref_pf).sum())}/{len(FOLDS)}",
                     verdict="ADDS" if (macro(a) - macro(single[REF]) > 0.003 and m5 > 0 and (pf > ref_pf).sum() >= len(FOLDS) - 1) else "no clear gain"))
print(f"{REF} + one other arm, vs {REF} alone:\n", pd.DataFrame(rows).round(4).to_string(index=False))
full = tuple(ARMS)
if len(ARMS) > 2:
    rows = []
    for n in ARMS:
        c = tuple(m for m in full if m != n); R = ens(c); a = aucs(R); Rf = ens(full); af = aucs(Rf)
        rows.append(dict(drop=n, macro=macro(a), d_macro_vs_full=macro(a) - macro(af), key4=key4(a), d_key4_vs_full=key4(a) - key4(af)))
    print(f"\nleave-one-out from the full ensemble ({macro(aucs(ens(full))):.4f} macro). A big negative delta = that member matters:\n", pd.DataFrame(rows).round(4).to_string(index=False))

convnext + one other arm, vs convnext alone:
         add  d_macro          macro_CI  d_key4           key4_CI folds_better       verdict
   baseline  -0.0129 [-0.0151,-0.0110] -0.0207 [-0.0241,-0.0172]          0/5 no clear gain
      v4mix  -0.0141 [-0.0163,-0.0122] -0.0198 [-0.0230,-0.0167]          0/5 no clear gain
v6convnext2   0.0043 [+0.0034,+0.0053]  0.0066 [+0.0049,+0.0083]          5/5          ADDS
      v6tok  -0.0098 [-0.0115,-0.0082] -0.0168 [-0.0195,-0.0139]          0/5 no clear gain

leave-one-out from the full ensemble (0.8062 macro). A big negative delta = that member matters:
        drop  macro  d_macro_vs_full   key4  d_key4_vs_full
   convnext 0.7962          -0.0100 0.7934         -0.0136
   baseline 0.8119           0.0057 0.8156          0.0087
      v4mix 0.8123           0.0061 0.8149          0.0079
v6convnext2 0.7949          -0.0113 0.7917         -0.0152
      v6tok 0.8095           0.0033 0.8130          0.0060


In [5]:
# ---- greedy weights (with replacement), checked honestly: choose on 4 folds, score on the 5th ----
def greedy(train_mask, steps=12):
    chosen, cur = [], None
    for _ in range(steps):
        best = None
        for n in ARMS:
            R = (RK[n] if cur is None else (cur * len(chosen) + RK[n]) / (len(chosen) + 1))
            s = macro(aucs(R, np.where(train_mask)[0]))
            if best is None or s > best[0]: best = (s, n, R)
        chosen.append(best[1]); cur = best[2]
    return chosen, cur
rows = []
for k in FOLDS:
    ch, R = greedy(fold != k)
    te = np.where(fold == k)[0]
    rows.append(dict(held_out_fold=k, greedy=macro(aucs(R, te)), ref=macro(aucs(RK[REF], te)), equal_all=macro(aucs(ens(full), te)),
                     weights=dict(pd.Series(ch).value_counts())))
G = pd.DataFrame(rows); print("held-out-fold macro (selection never saw that fold):\n", G.round(4).to_string(index=False))
print(f"\nmean: greedy {G.greedy.mean():.4f} | {REF} alone {G.ref.mean():.4f} | equal-weight all {G.equal_all.mean():.4f}")
ch, Rg = greedy(np.ones(N, bool)); print("greedy weights on all folds:", dict(pd.Series(ch).value_counts()), "| macro", round(macro(aucs(Rg)), 4), "(optimistic: chosen on the same data)")

held-out-fold macro (selection never saw that fold):
  held_out_fold  greedy    ref  equal_all                           weights
             0  0.8352 0.8328     0.8170 {'v6convnext2': 7, 'convnext': 5}
             1  0.8190 0.8149     0.8002 {'v6convnext2': 7, 'convnext': 5}
             2  0.8247 0.8206     0.8110 {'v6convnext2': 7, 'convnext': 5}
             3  0.8288 0.8232     0.8107 {'v6convnext2': 7, 'convnext': 5}
             4  0.8345 0.8276     0.8114 {'v6convnext2': 6, 'convnext': 6}

mean: greedy 0.8284 | convnext alone 0.8238 | equal-weight all 0.8101
greedy weights on all folds: {'v6convnext2': np.int64(7), 'convnext': np.int64(5)} | macro 0.8241 (optimistic: chosen on the same data)


In [6]:
# ---- per label: reference vs best subset vs full, plus gold-58 sanity ----
best = T.iloc[0].members.split(" + ")
P = pd.DataFrame({REF: single[REF], "best: " + " + ".join(best): aucs(ens(best)), "full": aucs(ens(full))}, index=LAB)
P.loc["MACRO"] = P.loc[LAB].mean(); P.loc["KEY-4"] = P.loc[KEY4].mean(); print(P.round(3).to_string())
if gold.sum() > 10:
    from sklearn.metrics import roc_auc_score
    def gm(R):
        v = [roc_auc_score((Y[gold, j] >= 0.5).astype(int), R[gold, j]) for j in range(len(LAB)) if len({*(Y[gold, j] >= 0.5)}) == 2]; return np.mean(v)
    print(f"\ngold ({int(gold.sum())} studies, CIs about +/-0.1, NOT decisive): " + " | ".join(f"{n} {gm(RK[n]):.3f}" for n in ARMS) + f" | best-subset {gm(ens(best)):.3f} | full {gm(ens(full)):.3f}")
print("""
HOW TO READ THIS
- Only trust differences larger than ~0.003 pooled macro AND a 90% CI that excludes 0 AND >= 4/5 folds improving.
- The held-out-fold greedy check is the honest one for weights. If it does not beat equal-weight, use equal weights.
- Seed noise on one fold is ~0.01, so per-fold deltas below that are not informative alone.
- Dropping an arm that adds nothing also saves inference time (~130 s per extra model on 1300 studies).""")

                  convnext  best: convnext + v6convnext2   full
ACL                  0.853                         0.860  0.845
MCL                  0.797                         0.801  0.771
Medial Meniscus      0.881                         0.887  0.841
Lateral Meniscus     0.786                         0.795  0.771
Medial OA            0.860                         0.860  0.852
Lateral OA           0.777                         0.777  0.772
PF OA                0.727                         0.727  0.735
Effusion             0.840                         0.845  0.837
Synovitis            0.791                         0.793  0.786
Baker's              0.850                         0.857  0.825
Contusion            0.824                         0.831  0.803
Fracture             0.851                         0.856  0.837
MACRO                0.820                         0.824  0.806
KEY-4                0.829                         0.836  0.807

gold (58 studies, CIs about +/-0.1, NOT